# Screw-mask test

The ruler curves suggest that the four nominally repeated blade-edge phases are not identical. One possible cause is the asymmetric set of screws around the rotating hub. This notebook tests that hypothesis before changing any experiment code or source data.

Because the screws rotate, three fixed image-space circles would only remove them at one wheel angle. The first mask candidate is therefore a **fixed central exclusion disk** large enough to cover the complete screw trajectories. Its centre and radius remain manual parameters.


## 1. Select a recording and adjust the mask

Start with F1 because its long rotation gives the clearest phase structure. Adjust `MASK_RADIUS_PX` after inspecting the central crop below. `MASK_CENTER_OFFSET_X/Y` should normally remain zero because the centre comes from the calibrated mask ellipse.


In [2]:
from pathlib import Path

import cv2
import h5py
import numpy as np
import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots

PROJECT_ROOT = Path("C:/Users/cxm3593/Academic/Workspace/EventSimilarityAnalysis")
TRIALS_ROOT = PROJECT_ROOT.parent / "EventCamCalib/output/trials"
TRIAL = "optical_chopper_data_f1"
PERIOD_INDEX = 0

SENSOR_WIDTH = 1280
SENSOR_HEIGHT = 720
MASK_CENTER_OFFSET_X = 0.0
MASK_CENTER_OFFSET_Y = 0.0
MASK_RADIUS_PX = 55.0   # First estimate; adjust manually.
CROP_RADIUS_PX = 250.0
VISUALIZATION_PHASE = 0.50  # Fraction through the selected rotation.

trial_dir = TRIALS_ROOT / TRIAL
result = yaml.safe_load((trial_dir / "result.yaml").read_text(encoding="utf-8"))
period_record = yaml.safe_load(
    (trial_dir / "rotation_period.yaml").read_text(encoding="utf-8")
)
rotation_period_us = int(round(period_record["rotation_period_us"]))
calibrated_centre = np.asarray(
    result["masked_events"]["mask_ellipse"]["center"], dtype=float
)
mask_centre = calibrated_centre + np.asarray([
    MASK_CENTER_OFFSET_X, MASK_CENTER_OFFSET_Y
])
real_path = trial_dir / "final_masked_real.h5"
warped_video_path = trial_dir / "frame_warped.avi"

print(f"Trial: {TRIAL}")
print(f"Full rotation: {rotation_period_us:,} us")
print(f"Calibrated centre: ({calibrated_centre[0]:.2f}, {calibrated_centre[1]:.2f}) px")
print(f"Candidate exclusion disk: centre=({mask_centre[0]:.2f}, {mask_centre[1]:.2f}), radius={MASK_RADIUS_PX:.1f} px")


Trial: optical_chopper_data_f1
Full rotation: 1,961,623 us
Calibrated centre: (492.54, 379.77) px
Candidate exclusion disk: centre=(492.54, 379.77), radius=55.0 px


## 2. Load one complete rotation efficiently

The HDF5 timestamps are sorted. Binary search locates the selected rotation without loading the complete 15-second recording. No source file is modified.


In [3]:
def h5_lower_bound(dataset, timestamp_us):
    low, high = 0, len(dataset)
    while low < high:
        middle = (low + high) // 2
        if int(dataset[middle]["t"]) < timestamp_us:
            low = middle + 1
        else:
            high = middle
    return low

def read_rotation(path, period_index, period_us):
    with h5py.File(path, "r") as handle:
        dataset = handle["events"]
        stream_start_us = int(dataset[0]["t"])
        start_us = stream_start_us + period_index * period_us
        end_us = start_us + period_us
        low = h5_lower_bound(dataset, start_us)
        high = h5_lower_bound(dataset, end_us)
        return dataset[low:high], start_us

real_events, rotation_start_us = read_rotation(
    real_path, PERIOD_INDEX, rotation_period_us
)
dx = real_events["x"].astype(np.float32) - np.float32(mask_centre[0])
dy = real_events["y"].astype(np.float32) - np.float32(mask_centre[1])
inside_screw_region = dx * dx + dy * dy <= MASK_RADIUS_PX ** 2

removed_count = int(inside_screw_region.sum())
removed_fraction = 100.0 * removed_count / len(real_events)
print(f"Loaded {len(real_events):,} events from period {PERIOD_INDEX}.")
print(f"Candidate mask removes {removed_count:,} events ({removed_fraction:.2f}%).")


Loaded 2,868,382 events from period 0.
Candidate mask removes 156,085 events (5.44%).


## 3. Inspect the mask placement

This view uses one video-frame interval rather than a density histogram. The spatially calibrated video frame, the corresponding event frame, and the event frame after masking share the same pixel coordinates and crop. The red circle is the proposed exclusion boundary. Change `VISUALIZATION_PHASE`, the centre offsets, or the radius and then re-run sections 1–3.


In [4]:
def render_event_frame(events, width, height):
    """Render the latest event at each pixel without an external controller."""
    image_rgb = np.zeros((height, width, 3), dtype=np.uint8)
    if len(events) == 0:
        return image_rgb

    x = events["x"].astype(np.int64)
    y = events["y"].astype(np.int64)
    polarity = events["p"]
    valid = (x >= 0) & (x < width) & (y >= 0) & (y < height)
    x, y, polarity = x[valid], y[valid], polarity[valid]
    if len(x) == 0:
        return image_rgb

    # A pixel may fire more than once in the interval. Keep its latest event,
    # matching the controller's single-frame behaviour.
    flat_pixel = y * width + x
    _, index_from_end = np.unique(flat_pixel[::-1], return_index=True)
    latest = len(flat_pixel) - 1 - index_from_end
    colours = np.zeros((len(latest), 3), dtype=np.uint8)
    positive = polarity[latest] > 0
    colours[positive] = (255, 255, 255)  # positive: white
    colours[~positive] = (0, 0, 255)     # negative: blue
    image_rgb[y[latest], x[latest]] = colours
    return image_rgb


video = cv2.VideoCapture(str(warped_video_path))
if not video.isOpened():
    raise FileNotFoundError(f"Could not open calibrated video: {warped_video_path}")

video_fps = float(video.get(cv2.CAP_PROP_FPS))
video_frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
requested_time_us = rotation_start_us + VISUALIZATION_PHASE * rotation_period_us
frame_index = min(int(np.floor(requested_time_us * video_fps / 1e6)), video_frame_count - 1)
frame_start_us = int(round(frame_index * 1e6 / video_fps))
frame_end_us = int(round((frame_index + 1) * 1e6 / video_fps))
video.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
frame_read_ok, video_frame_bgr = video.read()
video.release()
if not frame_read_ok:
    raise RuntimeError(f"Could not read video frame {frame_index}")

frame_events = real_events[
    (real_events["t"] >= frame_start_us)
    & (real_events["t"] < frame_end_us)
]
frame_dx = frame_events["x"].astype(float) - mask_centre[0]
frame_dy = frame_events["y"].astype(float) - mask_centre[1]
frame_inside_screw_region = frame_dx**2 + frame_dy**2 <= MASK_RADIUS_PX**2
masked_frame_events = frame_events[~frame_inside_screw_region]

event_frame_rgb = render_event_frame(frame_events, SENSOR_WIDTH, SENSOR_HEIGHT)
masked_event_frame_rgb = render_event_frame(
    masked_frame_events, SENSOR_WIDTH, SENSOR_HEIGHT
)
full_images_rgb = [
    cv2.cvtColor(video_frame_bgr, cv2.COLOR_BGR2RGB),
    event_frame_rgb,
    masked_event_frame_rgb,
]

crop_x0 = max(0, int(np.floor(mask_centre[0] - CROP_RADIUS_PX)))
crop_x1 = min(SENSOR_WIDTH, int(np.ceil(mask_centre[0] + CROP_RADIUS_PX)))
crop_y0 = max(0, int(np.floor(mask_centre[1] - CROP_RADIUS_PX)))
crop_y1 = min(SENSOR_HEIGHT, int(np.ceil(mask_centre[1] + CROP_RADIUS_PX)))
cropped_images_rgb = [
    image[crop_y0:crop_y1, crop_x0:crop_x1] for image in full_images_rgb
]

mask_placement_figure = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "Spatially calibrated video frame",
        f"Event frame before masking ({len(frame_events):,})",
        f"Event frame after masking ({len(masked_frame_events):,})",
    ],
    horizontal_spacing=0.045,
)
for column, image_rgb in enumerate(cropped_images_rgb, start=1):
    mask_placement_figure.add_trace(
        go.Image(z=image_rgb, x0=crop_x0, y0=crop_y0, dx=1, dy=1),
        row=1, col=column,
    )
    mask_placement_figure.add_shape(
        type="circle",
        x0=mask_centre[0] - MASK_RADIUS_PX,
        x1=mask_centre[0] + MASK_RADIUS_PX,
        y0=mask_centre[1] - MASK_RADIUS_PX,
        y1=mask_centre[1] + MASK_RADIUS_PX,
        line=dict(color="#d62728", width=3),
        row=1, col=column,
    )
    mask_placement_figure.update_xaxes(
        title_text="x (px)", range=[crop_x0, crop_x1],
        constrain="domain", row=1, col=column,
    )
    mask_placement_figure.update_yaxes(
        title_text="y (px)", range=[crop_y1, crop_y0],
        scaleanchor=("x" if column == 1 else f"x{column}"),
        scaleratio=1, row=1, col=column,
    )

frame_duration_ms = (frame_end_us - frame_start_us) / 1e3
removed_from_frame = len(frame_events) - len(masked_frame_events)
mask_placement_figure.update_layout(
    title=(f"Central screw-region mask — {TRIAL}, period {PERIOD_INDEX}"
           f"<br><sup>video frame {frame_index}; {frame_start_us / 1e6:.6f}–"
           f"{frame_end_us / 1e6:.6f} s ({frame_duration_ms:.2f} ms); "
           f"radius {MASK_RADIUS_PX:.1f} px</sup>"),
    template="plotly_white", width=1320, height=510,
    margin=dict(l=55, r=30, t=110, b=55),
    showlegend=False,
)
print(f"Video frame: {frame_index} at {frame_start_us / 1e6:.6f}–{frame_end_us / 1e6:.6f} s")
print(f"Events in frame interval: {len(frame_events):,}")
print(f"Events after mask: {len(masked_frame_events):,} (removed {removed_from_frame:,})")
mask_placement_figure.show()


Video frame: 29 at 0.966667–1.000000 s
Events in frame interval: 54,864
Events after mask: 51,836 (removed 3,028)


## 4. Run the ruler test on the screw-masked real data

This first creates a derived trial whose `final_masked_real.h5` contains only events outside the accepted screw circle. It then passes that derived trial to the original, unchanged Test 2 implementation using the same F1 settings as the original whole-period run: one complete rotation, 5 ms comparison windows, one reference rotation, shifts from zero to one rotation, and all five metrics.

The source HDF5 is never modified. The derived trial has a mask-specific name and includes `screw_mask.yaml`, while ruler results are written normally under that derived trial's own output folder. Dataset creation and ruler execution have separate switches and are both off by default.


In [9]:
import shutil
import subprocess
import sys

RULER_WINDOW_US = 5_000
RULER_MAX_SHIFT_PERIODS = 1.0
RULER_METRICS = ["mmd_rbf15"]  # Fast development run; expand for the final run.
MASK_COPY_CHUNK_EVENTS = 1_000_000
BUILD_DERIVED_TRIAL = False
REBUILD_DERIVED_TRIAL = False
RUN_MASKED_RULER = False

def path_number(value):
    return f"{value:.2f}".replace("-", "m").replace(".", "p")

mask_id = (
    f"x{path_number(mask_centre[0])}_y{path_number(mask_centre[1])}"
    f"_r{path_number(MASK_RADIUS_PX)}"
)
DERIVED_TRIAL_NAME = f"{TRIAL}_screw_mask_{mask_id}"
DERIVED_TRIALS_ROOT = PROJECT_ROOT / "output/derived_trials"
DERIVED_TRIAL_DIR = DERIVED_TRIALS_ROOT / DERIVED_TRIAL_NAME
DERIVED_MASKED_REAL_PATH = DERIVED_TRIAL_DIR / "final_masked_real.h5"
DERIVED_MASK_MANIFEST_PATH = DERIVED_TRIAL_DIR / "screw_mask.yaml"
MASKED_RULER_OUTPUT_ROOT = (
    PROJECT_ROOT / "output/tests" / DERIVED_TRIAL_NAME / "test2_ruler"
)

def requested_mask_record():
    return {
        "derived_trial": DERIVED_TRIAL_NAME,
        "source_trial": TRIAL,
        "source_real_path": str(real_path.resolve()),
        "mask": {
            "type": "exclude_circle",
            "centre_x_px": float(mask_centre[0]),
            "centre_y_px": float(mask_centre[1]),
            "radius_px": float(MASK_RADIUS_PX),
        },
        "rotation_period_us": int(rotation_period_us),
    }

def manifest_matches_current_mask(record):
    if not record:
        return False
    mask = record.get("mask", {})
    recorded_circle = np.asarray([
        mask.get("centre_x_px", np.nan),
        mask.get("centre_y_px", np.nan),
        mask.get("radius_px", np.nan),
    ], dtype=float)
    current_circle = np.asarray([mask_centre[0], mask_centre[1], MASK_RADIUS_PX])
    return (
        record.get("source_trial") == TRIAL
        and Path(record.get("source_real_path", "")).resolve() == real_path.resolve()
        and np.allclose(recorded_circle, current_circle, atol=1e-6)
    )

def build_derived_screw_mask_trial(rebuild=False):
    existing_manifest = None
    if DERIVED_MASK_MANIFEST_PATH.exists():
        existing_manifest = yaml.safe_load(
            DERIVED_MASK_MANIFEST_PATH.read_text(encoding="utf-8")
        )
    if DERIVED_MASKED_REAL_PATH.exists() and not rebuild:
        if not manifest_matches_current_mask(existing_manifest):
            raise ValueError(
                "The derived HDF5 exists but its manifest does not match the "
                "current mask. Set REBUILD_DERIVED_TRIAL = True to replace it."
            )
        print(f"Reusing existing derived trial: {DERIVED_TRIAL_DIR}")
        return existing_manifest

    DERIVED_TRIAL_DIR.mkdir(parents=True, exist_ok=True)
    temporary_path = DERIVED_TRIAL_DIR / "final_masked_real.incomplete.h5"
    if temporary_path.exists():
        temporary_path.unlink()

    kept_total = 0
    removed_total = 0
    with h5py.File(real_path, "r") as source_file, h5py.File(temporary_path, "w") as output_file:
        source_events = source_file["events"]
        for key, value in source_file.attrs.items():
            output_file.attrs[key] = value
        create_options = {"chunks": (65_536,)}
        if source_events.compression is not None:
            create_options["compression"] = source_events.compression
            create_options["compression_opts"] = source_events.compression_opts
        if source_events.shuffle:
            create_options["shuffle"] = True
        if source_events.fletcher32:
            create_options["fletcher32"] = True
        output_events = output_file.create_dataset(
            "events", shape=(0,), maxshape=(None,),
            dtype=source_events.dtype, **create_options,
        )
        for key, value in source_events.attrs.items():
            output_events.attrs[key] = value

        total_events = len(source_events)
        for start in range(0, total_events, MASK_COPY_CHUNK_EVENTS):
            stop = min(start + MASK_COPY_CHUNK_EVENTS, total_events)
            events = source_events[start:stop]
            dx = events["x"].astype(np.float64) - mask_centre[0]
            dy = events["y"].astype(np.float64) - mask_centre[1]
            keep = dx * dx + dy * dy > MASK_RADIUS_PX**2
            selected = events[keep]
            new_total = kept_total + len(selected)
            output_events.resize((new_total,))
            output_events[kept_total:new_total] = selected
            kept_total = new_total
            removed_total += len(events) - len(selected)
            print(
                f"Processed {stop:,}/{total_events:,} events; "
                f"kept {kept_total:,}", end="\r"
            )
    print()

    temporary_path.replace(DERIVED_MASKED_REAL_PATH)
    shutil.copy2(trial_dir / "result.yaml", DERIVED_TRIAL_DIR / "result.yaml")
    shutil.copy2(
        trial_dir / "rotation_period.yaml",
        DERIVED_TRIAL_DIR / "rotation_period.yaml",
    )
    record = requested_mask_record()
    record.update({
        "source_event_count": int(kept_total + removed_total),
        "kept_event_count": int(kept_total),
        "removed_event_count": int(removed_total),
        "removed_event_percent": float(
            100.0 * removed_total / (kept_total + removed_total)
        ),
    })
    DERIVED_MASK_MANIFEST_PATH.write_text(
        yaml.safe_dump(record, sort_keys=False), encoding="utf-8"
    )
    print(f"Created derived trial: {DERIVED_TRIAL_DIR}")
    return record

print("Masked ruler configuration:")
print(f"  source trial: {TRIAL}")
print(f"  derived trial: {DERIVED_TRIAL_NAME}")
print(f"  exclusion circle: ({mask_centre[0]:.2f}, {mask_centre[1]:.2f}), "
      f"radius {MASK_RADIUS_PX:.2f} px")
print(f"  metrics: {', '.join(RULER_METRICS)}")
print(f"  derived data: {DERIVED_TRIAL_DIR}")
print(f"  ruler results: {MASKED_RULER_OUTPUT_ROOT}")

if BUILD_DERIVED_TRIAL:
    build_derived_screw_mask_trial(rebuild=REBUILD_DERIVED_TRIAL)
elif DERIVED_MASKED_REAL_PATH.exists():
    print("Derived HDF5 already exists; set BUILD_DERIVED_TRIAL = True to validate/reuse it.")
else:
    print("Derived HDF5 not created. Set BUILD_DERIVED_TRIAL = True.")

masked_ruler_command = [
    sys.executable, str(PROJECT_ROOT / "experiments/test2_ruler.py"),
    "--trials-dir", str(DERIVED_TRIALS_ROOT),
    "--trial", DERIVED_TRIAL_NAME,
    "--baseline-start-us", str(rotation_start_us),
    "--n-baseline-windows", "1",
    "--span-us", "period",
    "--window-us", str(RULER_WINDOW_US),
    "--max-shift-periods", str(RULER_MAX_SHIFT_PERIODS),
    "--events-per-comparison", "none",
    "--metrics", *RULER_METRICS,
]
if RUN_MASKED_RULER:
    if not DERIVED_MASKED_REAL_PATH.exists():
        raise FileNotFoundError(
            "Create the derived trial before starting the ruler test."
        )
    subprocess.run(masked_ruler_command, cwd=PROJECT_ROOT, check=True)
else:
    print("Ruler test not started. Set RUN_MASKED_RULER = True when ready.")


Masked ruler configuration:
  source trial: optical_chopper_data_f1
  derived trial: optical_chopper_data_f1_screw_mask_x492p54_y379p77_r55p00
  exclusion circle: (492.54, 379.77), radius 55.00 px
  metrics: mmd_rbf15
  derived data: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\derived_trials\optical_chopper_data_f1_screw_mask_x492p54_y379p77_r55p00
  ruler results: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\tests\optical_chopper_data_f1_screw_mask_x492p54_y379p77_r55p00\test2_ruler
Derived HDF5 already exists; set BUILD_DERIVED_TRIAL = True to validate/reuse it.


## 5. Compare the original and screw-masked ruler results

This overlays the original and screw-masked ruler curves on the same axes. One panel is shown for each metric available in both runs; the spread across the 5 ms window pairs remains visible as error bars. The shaded region marks shifts smaller than one comparison window, where the paired windows overlap.

The cell only combines runs with the same window, whole-period span, shift range, and metric. Incomplete or incompatible folders are ignored, and the stored circle must match the current mask parameters. Note that the older original run's `run_config.yaml` copied the stale raw `biased: true` setting, but `common.build_metrics` forced the estimator used for the calculation to `biased: false`.


In [12]:
import pandas as pd

RULER_METRIC_ORDER = [
    "mmd_rbf03", "mmd_rbf15", "mmd_rbf75", "swd", "chamfer"
]
RULER_METRIC_LABELS = {
    "mmd_rbf03": "MMD RBF-3",
    "mmd_rbf15": "MMD RBF-15",
    "mmd_rbf75": "MMD RBF-75",
    "swd": "SWD",
    "chamfer": "Chamfer",
}
RULER_ORIGINAL_COLOUR = "#2a78d6"
RULER_MASKED_COLOUR = "#d1352b"
RULER_OVERLAP_COLOUR = "#9aa0a6"


def newest_compatible_ruler_run(project_root, trial, required_metrics):
    root = project_root / "output/tests" / trial / "test2_ruler"
    if not root.exists():
        return None, None
    for folder in sorted((path for path in root.iterdir() if path.is_dir()), reverse=True):
        results_path = folder / "results.csv"
        config_path = folder / "run_config.yaml"
        if not results_path.exists() or not config_path.exists():
            continue
        run_config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
        parameters = run_config.get("parameters", {})
        run_metrics = set(run_config.get("metrics", []))
        if int(parameters.get("window_us", -1)) != RULER_WINDOW_US:
            continue
        if not parameters.get("span_is_whole_period", False):
            continue
        if not np.isclose(
            float(parameters.get("max_shift_periods", np.nan)),
            RULER_MAX_SHIFT_PERIODS,
        ):
            continue
        if not set(required_metrics).issubset(run_metrics):
            continue
        return folder, run_config
    return None, None


def plot_ruler_mask_comparison(
    original_results, masked_results, masked_run_config, mask_manifest,
    original_run_folder, masked_run_folder,
):
    original_metrics = set(original_results["metric"])
    masked_metrics = set(masked_results["metric"])
    metrics = [metric for metric in RULER_METRIC_ORDER
               if metric in original_metrics & masked_metrics]
    if not metrics:
        raise ValueError("The two ruler runs have no metrics in common.")
    window_us = int(masked_results["window_length_us"].iloc[0])
    rotation_us = int(masked_run_config["trial"]["rotation_period_us"])
    mask = mask_manifest["mask"]
    circle = [mask["centre_x_px"], mask["centre_y_px"], mask["radius_px"]]
    n_pairs = int(masked_results["n_windows"].iloc[0])

    figure = make_subplots(
        rows=len(metrics), cols=1, shared_xaxes=True,
        vertical_spacing=0.035,
        subplot_titles=[RULER_METRIC_LABELS[metric] for metric in metrics],
    )
    for row, metric in enumerate(metrics, start=1):
        original_block = original_results[
            original_results["metric"] == metric
        ].sort_values("shift_us")
        masked_block = masked_results[
            masked_results["metric"] == metric
        ].sort_values("shift_us")
        if not np.array_equal(
            original_block["shift_us"].to_numpy(),
            masked_block["shift_us"].to_numpy(),
        ):
            raise ValueError(f"Shift positions do not match for {metric}.")

        for block, name, colour, dash, marker_symbol in [
            (original_block, "Original real data", RULER_ORIGINAL_COLOUR, "dash", "circle-open"),
            (masked_block, "Screw-masked real data", RULER_MASKED_COLOUR, "solid", "diamond"),
        ]:
            figure.add_trace(go.Scatter(
                x=block["shift_us"], y=block["mean_distance"],
                name=name, mode="lines+markers", showlegend=(row == 1),
                line=dict(color=colour, width=2, dash=dash),
                marker=dict(size=4, symbol=marker_symbol),
                error_y=dict(
                    type="data", array=block["mean_sd_within_span"],
                    visible=True, thickness=0.8, width=0, color=colour,
                ),
                customdata=block[["mean_sd_within_span", "n_windows"]],
                hovertemplate=(
                    f"{name}<br>shift %{{x:,.0f}} us"
                    "<br>mean distance %{y:.5g}"
                    "<br>window-pair SD %{customdata[0]:.5g}"
                    "<br>window pairs %{customdata[1]:,.0f}<extra></extra>"
                ),
            ), row=row, col=1)
        figure.add_vrect(
            x0=0, x1=window_us, row=row, col=1,
            fillcolor=RULER_OVERLAP_COLOUR, opacity=0.16,
            line_width=0, layer="below",
        )
        if row == 1:
            figure.add_annotation(
                x=window_us / 2, y=1.0, yref="y domain",
                text="windows overlap", showarrow=False, yanchor="bottom",
                font=dict(size=11, color="#52514e"), row=row, col=1,
            )
        figure.update_yaxes(title_text="mean distance", row=row, col=1)

    figure.update_xaxes(title_text="temporal shift (us)", row=len(metrics), col=1)
    figure.update_layout(
        title=(
            f"Ruler test before and after screw masking - {TRIAL}"
            f"<br><sup>one {rotation_us / 1e6:.3f} s rotation; "
            f"{window_us / 1e3:g} ms windows; {n_pairs} window pairs per shift; "
            f"excluded circle ({circle[0]:.2f}, {circle[1]:.2f}), "
            f"radius {circle[2]:.2f} px</sup>"
        ),
        template="plotly_white", height=max(480, 190 * len(metrics)), width=950,
        margin=dict(l=85, r=30, t=145, b=65),
        legend=dict(
            orientation="h", x=0.5, xanchor="center",
            y=1.02, yanchor="bottom",
        ),
    )
    print(f"Original:     {original_run_folder}")
    print(f"Screw-masked: {masked_run_folder}")
    return figure


masked_ruler_folder, masked_ruler_config = newest_compatible_ruler_run(
    PROJECT_ROOT, DERIVED_TRIAL_NAME, RULER_METRICS
)
if masked_ruler_folder is None:
    print("No completed screw-mask ruler run was found. Build the derived trial and run Section 4 first.")
else:
    if not DERIVED_MASK_MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Missing mask manifest: {DERIVED_MASK_MANIFEST_PATH}")
    masked_ruler_manifest = yaml.safe_load(
        DERIVED_MASK_MANIFEST_PATH.read_text(encoding="utf-8")
    )
    manifest_mask = masked_ruler_manifest["mask"]
    configured_circle = np.asarray([
        manifest_mask["centre_x_px"],
        manifest_mask["centre_y_px"],
        manifest_mask["radius_px"],
    ], dtype=float)
    current_circle = np.asarray([mask_centre[0], mask_centre[1], MASK_RADIUS_PX])
    if not np.allclose(configured_circle, current_circle, atol=0.01):
        raise ValueError(
            "The newest completed run used a different screw mask. "
            f"Run: {configured_circle.tolist()}, current: {current_circle.tolist()}. "
            "Rerun Section 4 before plotting."
        )
    original_ruler_folder, original_ruler_config = newest_compatible_ruler_run(
        PROJECT_ROOT, TRIAL, RULER_METRICS
    )
    if original_ruler_folder is None:
        raise RuntimeError("No compatible original ruler run was found.")

    original_ruler_results = pd.read_csv(original_ruler_folder / "results.csv")
    masked_ruler_results = pd.read_csv(masked_ruler_folder / "results.csv")
    ruler_mask_comparison_figure = plot_ruler_mask_comparison(
        original_ruler_results, masked_ruler_results, masked_ruler_config,
        masked_ruler_manifest, original_ruler_folder, masked_ruler_folder,
    )
    ruler_mask_comparison_figure.show()


Original:     C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\tests\optical_chopper_data_f1\test2_ruler\20260824_192020_period_w5000us
Screw-masked: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\tests\optical_chopper_data_f1_screw_mask_x492p54_y379p77_r55p00\test2_ruler\20260828_193816_period_w5000us


## 6. Inspect one ruler window pair in 2D

This diagnostic shows the same reference and shifted 5 ms windows before and after screw masking. By default it selects the temporal shift where masking increased the mean ruler distance the most, then chooses an informative phase window containing many events inside the excluded circle. This is one example from the 393 window pairs, not a replacement for the averaged ruler result.

Set `SAMPLE_SHIFT_US` or `SAMPLE_PHASE_INDEX` to an integer to inspect a specific comparison. The view is cropped around the wheel centre; the Plotly controls can be used to zoom or pan further.


In [14]:
SAMPLE_METRIC = RULER_METRICS[0]
SAMPLE_SHIFT_US = None       # None selects the largest masked-minus-original increase.
SAMPLE_PHASE_INDEX = None    # None selects an informative 5 ms phase window.
SAMPLE_VIEW_RADIUS_PX = 260


def read_h5_event_range(path, start_us, end_us):
    with h5py.File(path, "r") as handle:
        dataset = handle["events"]
        low = h5_lower_bound(dataset, int(start_us))
        high = h5_lower_bound(dataset, int(end_us))
        return dataset[low:high]


def events_inside_circle(events, centre, radius_px):
    dx = events["x"].astype(np.float32) - np.float32(centre[0])
    dy = events["y"].astype(np.float32) - np.float32(centre[1])
    return dx * dx + dy * dy <= radius_px**2


metric_original = original_ruler_results[
    original_ruler_results["metric"] == SAMPLE_METRIC
][["shift_us", "mean_distance"]].rename(
    columns={"mean_distance": "original_distance"}
)
metric_masked = masked_ruler_results[
    masked_ruler_results["metric"] == SAMPLE_METRIC
][["shift_us", "mean_distance"]].rename(
    columns={"mean_distance": "masked_distance"}
)
distance_change = metric_original.merge(metric_masked, on="shift_us")
distance_change["increase"] = (
    distance_change["masked_distance"]
    - distance_change["original_distance"]
)
if SAMPLE_SHIFT_US is None:
    selected_shift_row = distance_change.loc[distance_change["increase"].idxmax()]
    selected_shift_us = int(selected_shift_row["shift_us"])
else:
    selected_shift_us = int(SAMPLE_SHIFT_US)
    match = distance_change[distance_change["shift_us"] == selected_shift_us]
    if match.empty:
        raise ValueError(
            f"Shift {selected_shift_us:,} us is not present in the ruler result."
        )
    selected_shift_row = match.iloc[0]

parameters = original_ruler_config["parameters"]
baseline_start_us = int(parameters["baseline_starts_us"][0])
span_us = int(parameters["span_length_us"])
window_us = int(parameters["window_us"])
phase_offsets_us = np.arange(0, span_us, window_us, dtype=np.int64)
reference_starts_us = baseline_start_us + phase_offsets_us
shifted_starts_us = reference_starts_us + selected_shift_us
reference_span_end_us = baseline_start_us + span_us
shifted_span_end_us = baseline_start_us + selected_shift_us + span_us

selection_events = read_h5_event_range(
    real_path, baseline_start_us, shifted_span_end_us
)
inside_times = selection_events["t"][
    events_inside_circle(selection_events, mask_centre, MASK_RADIUS_PX)
]
reference_ends_us = np.minimum(
    reference_starts_us + window_us, reference_span_end_us
)
shifted_ends_us = np.minimum(
    shifted_starts_us + window_us, shifted_span_end_us
)
reference_inside_counts = (
    np.searchsorted(inside_times, reference_ends_us)
    - np.searchsorted(inside_times, reference_starts_us)
)
shifted_inside_counts = (
    np.searchsorted(inside_times, shifted_ends_us)
    - np.searchsorted(inside_times, shifted_starts_us)
)
if SAMPLE_PHASE_INDEX is None:
    phase_index = int(np.argmax(reference_inside_counts + shifted_inside_counts))
else:
    phase_index = int(SAMPLE_PHASE_INDEX)
    if not 0 <= phase_index < len(phase_offsets_us):
        raise ValueError(
            f"SAMPLE_PHASE_INDEX must be between 0 and {len(phase_offsets_us) - 1}."
        )

reference_start_us = int(reference_starts_us[phase_index])
reference_end_us = int(reference_ends_us[phase_index])
shifted_start_us = int(shifted_starts_us[phase_index])
shifted_end_us = int(shifted_ends_us[phase_index])
original_reference = read_h5_event_range(
    real_path, reference_start_us, reference_end_us
)
masked_reference = read_h5_event_range(
    DERIVED_MASKED_REAL_PATH, reference_start_us, reference_end_us
)
original_shifted = read_h5_event_range(
    real_path, shifted_start_us, shifted_end_us
)
masked_shifted = read_h5_event_range(
    DERIVED_MASKED_REAL_PATH, shifted_start_us, shifted_end_us
)

panel_data = [
    (original_reference, reference_start_us),
    (masked_reference, reference_start_us),
    (original_shifted, shifted_start_us),
    (masked_shifted, shifted_start_us),
]
panel_titles = [
    f"Reference - original ({len(original_reference):,} events)",
    f"Reference - screw-masked ({len(masked_reference):,} events)",
    f"Shifted - original ({len(original_shifted):,} events)",
    f"Shifted - screw-masked ({len(masked_shifted):,} events)",
]
sample_window_figure = make_subplots(
    rows=2, cols=2, subplot_titles=panel_titles,
    horizontal_spacing=0.07, vertical_spacing=0.12,
)
polarity_styles = [
    (False, "Negative polarity", "#3274a1"),
    (True, "Positive polarity", "#e1812c"),
]
for panel_number, ((events, start_us), title) in enumerate(
    zip(panel_data, panel_titles), start=1
):
    row = 1 if panel_number <= 2 else 2
    column = 1 if panel_number % 2 else 2
    in_view = (
        (events["x"] >= mask_centre[0] - SAMPLE_VIEW_RADIUS_PX)
        & (events["x"] <= mask_centre[0] + SAMPLE_VIEW_RADIUS_PX)
        & (events["y"] >= mask_centre[1] - SAMPLE_VIEW_RADIUS_PX)
        & (events["y"] <= mask_centre[1] + SAMPLE_VIEW_RADIUS_PX)
    )
    visible_events = events[in_view]
    for positive, name, colour in polarity_styles:
        selected = visible_events[(visible_events["p"] > 0) == positive]
        sample_window_figure.add_trace(go.Scattergl(
            x=selected["x"], y=selected["y"], mode="markers",
            name=name, legendgroup=name, showlegend=(panel_number == 1),
            marker=dict(size=3, color=colour, opacity=0.65),
            customdata=(selected["t"] - start_us) / 1e3,
            hovertemplate=(
                "x %{x:.0f}, y %{y:.0f}"
                "<br>time in window %{customdata:.3f} ms<extra></extra>"
            ),
        ), row=row, col=column)
    sample_window_figure.add_shape(
        type="circle",
        x0=mask_centre[0] - MASK_RADIUS_PX,
        x1=mask_centre[0] + MASK_RADIUS_PX,
        y0=mask_centre[1] - MASK_RADIUS_PX,
        y1=mask_centre[1] + MASK_RADIUS_PX,
        line=dict(color="#d62728", width=2, dash="dash"),
        row=row, col=column,
    )
    sample_window_figure.update_xaxes(
        title_text="x (px)",
        range=[mask_centre[0] - SAMPLE_VIEW_RADIUS_PX,
               mask_centre[0] + SAMPLE_VIEW_RADIUS_PX],
        constrain="domain", row=row, col=column,
    )
    x_axis_name = "x" if panel_number == 1 else f"x{panel_number}"
    sample_window_figure.update_yaxes(
        title_text="y (px)",
        range=[mask_centre[1] + SAMPLE_VIEW_RADIUS_PX,
               mask_centre[1] - SAMPLE_VIEW_RADIUS_PX],
        scaleanchor=x_axis_name, scaleratio=1, row=row, col=column,
    )

sample_window_figure.update_layout(
    title=(
        f"Example ruler window pair - {SAMPLE_METRIC}"
        f"<br><sup>phase window {phase_index}; "
        f"{window_us / 1e3:g} ms windows; shift {selected_shift_us:,} us; "
        f"mean-distance increase {selected_shift_row['increase']:.5f}</sup>"
    ),
    template="plotly_white", width=1080, height=850,
    margin=dict(l=60, r=30, t=135, b=95),
    legend=dict(
        orientation="h", x=0.5, xanchor="center",
        y=-0.08, yanchor="top",
    ),
)
print(f"Selected shift: {selected_shift_us:,} us")
print(f"Selected phase window: {phase_index} of {len(phase_offsets_us)}")
print(
    f"Events removed: reference {len(original_reference) - len(masked_reference):,}; "
    f"shifted {len(original_shifted) - len(masked_shifted):,}"
)
sample_window_figure.show()


Selected shift: 1,300,000 us
Selected phase window: 372 of 393
Events removed: reference 553; shifted 504
